In [ ]:
!pip install scikeras

# Uninstall the current scikit-learn version
!pip uninstall scikit-learn -y

# Install a compatible version of scikit-learn (e.g., 1.4.2)
!pip install scikit-learn==1.4.2

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


In [126]:
# Read training data
train_df = pd.read_csv('train.csv')

In [147]:
# Get decision tree feature importance
match_df = train_df.drop(columns=['color', 'car_name', 'map_code']).groupby(['match_id', 'rank']).mean().reset_index()

X = match_df.drop(columns=['match_id', 'rank'])
y = match_df['rank']

In [148]:
le = LabelEncoder()
y = le.fit_transform(y)
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [169]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

In [170]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(), X.columns)
    ],
    remainder='passthrough'
)

In [171]:
es = EarlyStopping(monitor='val_loss', patience=2)

In [172]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(len(X.columns),)))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(6, activation='softmax')) # Need output node per rank
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=100,
    validation_split=0.2,
    callbacks=[es]
)

In [173]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', model)
    ]
).fit(X_train, y_train)

Epoch 1/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5155 - loss: 1.0973 - val_accuracy: 0.5436 - val_loss: 1.0371
Epoch 2/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5552 - loss: 1.0008 - val_accuracy: 0.5461 - val_loss: 1.0345
Epoch 3/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5648 - loss: 0.9766 - val_accuracy: 0.5599 - val_loss: 0.9925
Epoch 4/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5718 - loss: 0.9605 - val_accuracy: 0.5469 - val_loss: 1.0259
Epoch 5/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5834 - loss: 0.9425 - val_accuracy: 0.5552 - val_loss: 0.9961


In [174]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [175]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[ 285,    0,    4,   24,   17,  112],
       [   2, 2462,  980,    3,   77,    3],
       [   4,  717, 2603,  100,  719,    6],
       [  54,    7,  143, 2049,  770,  728],
       [  23,   68, 1218, 1009, 2089,   92],
       [ 210,    0,    5,  364,   58, 1067]])

In [176]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)

array([[ 164,    0,    5,   25,   12,   89],
       [   0, 1607,  677,    3,   64,    0],
       [   2,  551, 1596,   83,  531,    4],
       [  47,    4,  116, 1249,  576,  509],
       [  13,   63,  841,  715, 1303,   64],
       [ 156,    1,    5,  242,   47,  685]])

In [177]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       0.49      0.64      0.56       442
           1       0.76      0.70      0.73      3527
           2       0.53      0.63      0.57      4149
           3       0.58      0.55      0.56      3751
           4       0.56      0.46      0.51      4499
           5       0.53      0.63      0.57      1704

    accuracy                           0.58     18072
   macro avg       0.57      0.60      0.58     18072
weighted avg       0.59      0.58      0.58     18072

              precision    recall  f1-score   support

           0       0.43      0.56      0.48       295
           1       0.72      0.68      0.70      2351
           2       0.49      0.58      0.53      2767
           3       0.54      0.50      0.52      2501
           4       0.51      0.43      0.47      2999
           5       0.51      0.60      0.55      1136

    accuracy                           0.55     12049
   macro avg       0.53

In [178]:
test_df = pd.read_csv('test.csv')

In [179]:
match_test_df = test_df.drop(columns=['color', 'car_name', 'map_code']).groupby(['match_id']).mean().reset_index()

In [180]:
y_pred = pipe.predict(match_test_df[X.columns])

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [181]:
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [182]:
converter = { 0: 1, 5: 2, 3: 3, 4: 4, 2: 5, 1: 6 }

y_pred = pd.Series(y_pred).map(converter)

In [183]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})
submission.head()

,match_id,rank
0,30121,5
1,30122,3
2,30123,4
3,30124,4
4,30125,6


In [184]:
submission.to_csv('keras_mean_all_features_3-128_relu_submission_2.csv', index=False)

In [185]:
submission.shape

(2500, 2)